# AIBackends - local audio transcription with Whisper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb)

Transcribe audio locally with the built-in `AudioIngestor` + `WhisperTranscriber`
workflow steps (`aibackends[audio]`, powered by faster-whisper). Mirrors
`examples/workflows/audio_transcribe.py`, then chains the transcript into a PII
redaction step to show how audio fits into larger pipelines.

`WhisperTranscriber` loads faster-whisper with `device="auto"`, so it uses CUDA
automatically on a GPU runtime and CPU otherwise.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
%pip install -q "aibackends[audio,pii]>=0.8.1"

# If a later import fails with a transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [2]:
import aibackends

# aibackends accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch

    DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print("aibackends", aibackends.__version__)
print("device:", DEVICE)

aibackends 0.8.1
device: cpu


In [3]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://raw.githubusercontent.com/donvito/aibackends/main/examples/data"
DATA_DIR = Path("aibackends_data")


def fetch(relative_path: str) -> Path:
    """Download a sample file from the aibackends examples once and return its path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        try:
            urlretrieve(f"{DATA_URL}/{relative_path}", partial)
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    return path

## 1. Transcribe an MP3

In [4]:
import time

from aibackends.steps.ingest import AudioIngestor
from aibackends.steps.process import WhisperTranscriber
from aibackends.workflows import Pipeline

WHISPER_MODEL = "small" if DEVICE == "gpu" else "base"


class AudioTranscriptionWorkflow(Pipeline):
    steps = [AudioIngestor(), WhisperTranscriber(model_name=WHISPER_MODEL)]


audio_path = fetch("audio/audio1.mp3")
t = time.perf_counter()
result = AudioTranscriptionWorkflow().run(audio_path)
print(f"whisper-{WHISPER_MODEL} in {time.perf_counter() - t:.1f}s\n")
print(result["transcript"])

whisper-base in 26.3s

Black Hole Stars may have been the largest stars that ever existed. They burned
brighter than galaxies and were larger than any star today or that could ever
exist in the future. But besides their scale, what makes them special and weird
is the deep inside, they were occupied by a cosmic parasite, an endlessly hungry
black hole. How is that even possible?
Black Hole Stars take the weirdness of black holes and go beyond to break
everything we know about how stars form and grow. They were only possible
during a short window of time in the early universe, but if they existed,
they would solve one of the largest mysteries of cosmology. Black Hole Stars
were excessive anyway you look at them. The most massive stars today may have
about 300 solar masses. A black hole star had up to ten million solar masses
of nearly pure hydrogen. Let's take a moment to look at what this means
visually. The Sun, Wesson, L. Pegasi, the largest star and finally the black hole
star. Its s

## 2. Transcribe, then redact PII in one pipeline

In [5]:
from aibackends.steps.enrich import PIIRedactor


class RedactedTranscriptWorkflow(Pipeline):
    steps = [
        AudioIngestor(),
        WhisperTranscriber(model_name=WHISPER_MODEL),
        PIIRedactor(backend="gliner", labels=["person_name", "location", "organization"]),
    ]


redacted = RedactedTranscriptWorkflow().run(audio_path)
print(redacted["transcript"])
print(redacted["pii_redaction"].redaction_map)

Black Hole Stars may have been the largest stars that ever existed. They burned
brighter than galaxies and were larger than any star today or that could ever
exist in the future. But besides their scale, what makes them special and weird
is the deep inside, they were occupied by a cosmic parasite, an endlessly hungry
black hole. How is that even possible?
Black Hole Stars take the weirdness of black holes and go beyond to break
everything we know about how stars form and grow. They were only possible
during a short window of time in the early universe, but if they existed,
they would solve one of the largest mysteries of cosmology. Black Hole Stars
were excessive anyway you look at them. The most massive stars today may have
about 300 solar masses. A black hole star had up to ten million solar masses
of nearly pure hydrogen. Let's take a moment to look at what this means
visually. The Sun, [LOCATION_1], [ORGANIZATION_2], the largest star and finally the black hole
star. Its scale is be